# Feature Engineering, Feature Selection & Encoding

**Objective:** transform the cleaned hotel-booking dataset into a documented set of
candidate predictors for the ML data-preparation stage. This notebook creates
derived features, removes leakage/identifier/unavailable fields, records encoding
strategy, and exports the selected feature dataset.

**Scope boundary:** no train/test split, fitted encoder, scaler, model training or
prediction is performed here. Learned transformations must be fitted later using
training data only.

### 1. Load cleaned dataset

The input is the validated cleaned dataset from `03_Data_Preprocessing`. CSV dtype
metadata is restored with the shared loader before feature decisions are made.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/processed/hotel_booking_cleaned.csv').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.clean_hotel_bookings import load_cleaned
from scripts.feature_engineering_hotel_bookings import (
    OUTPUT_DIR, FEATURED_PATH, ENGINEERED_COLUMNS,
    prepare_feature_engineering_outputs, export_feature_engineering_outputs,
)

pd.set_option('display.max_rows', 120)
cleaned = load_cleaned()
print(f'Cleaned input shape: {cleaned.shape[0]:,} rows x {cleaned.shape[1]} columns')
display(cleaned.head())

Cleaned input shape: 119,205 rows x 36 columns


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,name,email,phone-number,credit_card
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Ernest Barnes,Ernest.Barnes31@outlook.com,669-792-1661,************4322
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,Andrea Baker,Andrea_Baker94@aol.com,858-637-6955,************9157
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Rebecca Parker,Rebecca_Parker@comcast.net,652-885-2745,************3734
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,Laura Murray,Laura_M@gmail.com,364-656-8427,************5677
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,Linda Hines,LHines@verizon.com,713-226-5883,************5498


### 2. Review available features

The review separates target, numerical fields, categorical fields, date fields and
personal identifiers. High-cardinality fields are not encoded here; only their
later encoding strategy is documented.

In [2]:
result = prepare_feature_engineering_outputs(cleaned)
display(result['feature_types'])

,column,role,dtype,missing,unique_values
0,hotel,categorical,category,0,2
1,is_canceled,target,int8,0,2
2,lead_time,numerical,int64,0,479
3,arrival_date_year,numerical,int64,0,3
4,arrival_date_month,categorical,category,0,12
5,arrival_date_week_number,numerical,int64,0,53
6,arrival_date_day_of_month,numerical,int64,0,31
7,stays_in_weekend_nights,numerical,int64,0,17
8,stays_in_week_nights,numerical,int64,0,33
9,adults,numerical,int64,0,14


### 3. Leakage and identifier decisions

`reservation_status` and `reservation_status_date` are removed because they reveal
information only known after the cancellation outcome. Personal fields are removed
because they identify guests and do not provide appropriate modelling signal.

The project prediction point is the moment the initial reservation is created.
`assigned_room_type`, `booking_changes` and `days_in_waiting_list` are excluded
because they are likely known or accumulated after that point. Parking spaces and
special requests are retained under the team assumption that those requests are
recorded at initial reservation time; this assumption should be revisited if the
source-system timing is later clarified.

In [3]:
decisions = result['feature_decisions']
display(decisions.loc[decisions['action'].isin(['exclude', 'target'])])

,column,action,reason
1,is_canceled,target,Prediction outcome; kept separately for later ...
20,assigned_room_type,exclude,Actual assigned room may be known after reserv...
21,booking_changes,exclude,Counts changes that can happen after the initi...
25,days_in_waiting_list,exclude,Completed waiting duration is not known at ini...
30,reservation_status,exclude,Directly encodes the final booking outcome.
31,reservation_status_date,exclude,Recorded after the final booking outcome/statu...
32,name,exclude,Personal identifier; not useful or appropriate...
33,email,exclude,Personal identifier; not useful or appropriate...
34,phone-number,exclude,Personal identifier; not useful or appropriate...
35,credit_card,exclude,Personal identifier; not useful or appropriate...


### 4. Feature engineering

The engineered features are based on EDA findings and business meaning:

- `total_stay`: total booked nights.
- `total_guests`: total party size.
- `has_children` and `has_babies`: family-composition indicators.
- `previous_total_bookings`: previous booking-history volume.
- `previous_cancellation_rate`: previous cancellations divided by previous total
  bookings, with zero history mapped to 0.

In [4]:
print('Engineered columns:')
display(pd.DataFrame({'engineered_feature': ENGINEERED_COLUMNS}))
display(result['featured'][['is_canceled', *ENGINEERED_COLUMNS]].head())

Engineered columns:


,engineered_feature
0,total_stay
1,total_guests
2,has_children
3,has_babies
4,previous_total_bookings
5,previous_cancellation_rate


,is_canceled,total_stay,total_guests,has_children,has_babies,previous_total_bookings,previous_cancellation_rate
0,0,0,2,0,0,0,0.0
1,0,0,2,0,0,0,0.0
2,0,1,1,0,0,0,0.0
3,0,1,1,0,0,0,0.0
4,0,2,2,0,0,0,0.0


### 5. Evaluate engineered features

Evaluation uses distribution summaries, class-wise means and absolute correlation
with the binary target. Some engineered features have weak individual linear
correlation with cancellation, so they are retained as candidate predictors because
they have meaningful business interpretations and may contribute through non-linear
relationships or interactions. Final usefulness belongs to model development.

In [5]:
display(result['engineered_evaluation'].round(6))

,feature,dtype,unique_values,min,max,mean_not_canceled,mean_canceled,mean_difference_canceled_minus_not,absolute_correlation_with_target,decision,decision_note
0,total_stay,int64,42,0.0,69.0,3.389975,3.487702,0.097728,0.018579,retain_candidate,Retained as a candidate predictor because it h...
1,total_guests,int64,14,1.0,55.0,1.946474,2.013169,0.066695,0.044811,retain_candidate,Retained as a candidate predictor because it h...
2,has_children,int8,2,0.0,1.0,0.072844,0.070732,-0.002112,0.003944,retain_candidate,Retained as a candidate predictor because it h...
3,has_babies,int8,2,0.0,1.0,0.009999,0.003779,-0.006220,0.034386,retain_candidate,Retained as a candidate predictor because it h...
4,previous_total_bookings,int64,77,0.0,78.0,0.218797,0.233578,0.014781,0.003904,retain_candidate,Retained as a candidate predictor because it h...
5,previous_cancellation_rate,float64,111,0.0,1.0,0.001707,0.132602,0.130895,0.291841,retain_candidate,Retained as a candidate predictor because it h...


### 6. Feature selection

The selected dataset keeps cleaned original predictors that are available at the
initial booking point plus retained engineered features. Confirmed leakage,
identifiers and clearly post-booking fields are removed. Original component fields
such as weekday/weekend nights, guest composition and previous-history counts are
retained with their engineered summaries because the summaries do not fully replace
the raw information.

In [6]:
display(result['feature_decisions'])
display(result['selected_features'])
print(f"Selected predictor count: {len(result['selected_features'])}")

,column,action,reason
0,hotel,retain_original,Cleaned source feature retained for later mode...
1,is_canceled,target,Prediction outcome; kept separately for later ...
2,lead_time,retain_original,Cleaned source feature retained for later mode...
3,arrival_date_year,retain_original,Cleaned source feature retained for later mode...
4,arrival_date_month,retain_original,Cleaned source feature retained for later mode...
5,arrival_date_week_number,retain_original,Cleaned source feature retained for later mode...
6,arrival_date_day_of_month,retain_original,Cleaned source feature retained for later mode...
7,stays_in_weekend_nights,retain_original_component,Retained with total_stay because weekend and w...
8,stays_in_week_nights,retain_original_component,Retained with total_stay because weekend and w...
9,adults,retain_original_component,Retained with total_guests because party compo...


,selected_feature
0,hotel
1,lead_time
2,arrival_date_year
3,arrival_date_month
4,arrival_date_week_number
5,arrival_date_day_of_month
6,stays_in_weekend_nights
7,stays_in_week_nights
8,adults
9,children


Selected predictor count: 32


### 7. Encoding and high-cardinality strategy

This section documents the intended transformation strategy only. The later ML
data-preparation notebook must fit encoders on training rows only. `country`,
`agent` and `company` are high-cardinality nominal fields, so rare-category grouping
should be learned from training data rather than from the full cleaned dataset.

In [7]:
display(result['encoding_strategy'])

,column,encoding_strategy,unique_values_full_cleaned_data,reason
0,hotel,one_hot,2,Nominal categorical field; one-hot encoding av...
1,lead_time,numeric_passthrough_or_scale_later,479,Keep numerical value; scaling choice belongs t...
2,arrival_date_year,numeric_passthrough_or_scale_later,3,Keep numerical value; scaling choice belongs t...
3,arrival_date_month,one_hot,12,Nominal categorical field; one-hot encoding av...
4,arrival_date_week_number,numeric_passthrough_or_scale_later,53,Keep numerical value; scaling choice belongs t...
5,arrival_date_day_of_month,numeric_passthrough_or_scale_later,31,Keep numerical value; scaling choice belongs t...
6,stays_in_weekend_nights,numeric_passthrough_or_scale_later,17,Keep numerical value; scaling choice belongs t...
7,stays_in_week_nights,numeric_passthrough_or_scale_later,33,Keep numerical value; scaling choice belongs t...
8,adults,numeric_passthrough_or_scale_later,14,Keep numerical value; scaling choice belongs t...
9,children,numeric_passthrough_or_scale_later,5,Keep numerical value; scaling choice belongs t...


### 8. Export selected feature dataset and reports

`data/processed/hotel_booking_features.csv` is the handoff dataset for ML data
preparation. It contains `is_canceled` plus selected predictor columns. Supporting
CSV files document feature types, engineered-feature evaluation, selection decisions,
final selected features, encoding strategy and validation.

In [8]:
export_feature_engineering_outputs(result)
display(result['validation'])
print(f"Saved selected feature dataset to {FEATURED_PATH.relative_to(ROOT)}")
print(f"Saved feature-engineering reports to {OUTPUT_DIR.relative_to(ROOT)}")

,check,passed
0,Target column is present,True
1,All engineered columns selected,True
2,Confirmed leakage columns absent,True
3,PII columns absent,True
4,Prediction-time uncertain columns absent,True
5,No missing values in selected dataset,True
6,Target remains binary,True
7,Row count preserved from cleaned dataset,True
8,Selected column names are unique,True


Saved selected feature dataset to data\processed\hotel_booking_features.csv
Saved feature-engineering reports to 04_Feature_Engineering


### 9. Conclusion

Feature engineering and selection are complete for this stage. The next stage should
load `hotel_booking_features.csv`, separate `X` and `y`, create train/test
partitions, and fit encoding/scaling transformations on training data only.